# DLGenAI Project — Milestone 5

This notebook covers:

- Softmax inference
- DeBERTa and RoBERTa model ensembling
- Weighted probability averaging
- Kaggle Top-3 submission generation
- Test-Time Augmentation
- Confidence comparison
- MAP@3 evaluation


In [1]:
!pip install -q transformers sentencepiece accelerate


In [2]:
import os
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = ""

device = torch.device("cpu")

print("Using device:", device)

Using device: cpu


In [3]:
import os
import glob
import random
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    set_seed
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

label2id = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
id2label = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}
option_letters = ["A", "B", "C", "D", "E"]

print("Device:", device)


Device: cpu


## Dataset and checkpoint paths

The notebook searches common Kaggle and local locations automatically.

The two fine-tuned checkpoints must contain a five-label sequence-classification model:

- DeBERTa checkpoint based on `microsoft/deberta-v3-small`
- RoBERTa checkpoint based on `roberta-base`

Edit the two checkpoint variables manually if automatic detection does not find them.


In [4]:
import os
import pandas as pd

def first_existing(paths):
    for path in paths:
        if os.path.exists(path):
            return path
    return None

train_candidates = [
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    "/kaggle/input/smart-mcq-solver-challenge/train.csv",
    "/kaggle/working/train.csv",
    "/content/train.csv",
    "train.csv"
]

test_candidates = [
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv",
    "/kaggle/input/smart-mcq-solver-challenge/test.csv",
    "/kaggle/working/test.csv",
    "/content/test.csv",
    "test.csv"
]

train_path = first_existing(train_candidates)
test_path = first_existing(test_candidates)

if train_path is None:
    raise FileNotFoundError("train.csv was not found.")

if test_path is None:
    raise FileNotFoundError("test.csv was not found.")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

deberta_checkpoint = "microsoft/deberta-v3-small"
roberta_checkpoint = "roberta-base"

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("DeBERTa:", deberta_checkpoint)
print("RoBERTa:", roberta_checkpoint)

Train shape: (2000, 8)
Test shape: (500, 7)
DeBERTa: microsoft/deberta-v3-small
RoBERTa: roberta-base


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

deberta_tokenizer = AutoTokenizer.from_pretrained(
    deberta_checkpoint
)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_checkpoint,
    num_labels=5,
    ignore_mismatched_sizes=True
)

roberta_tokenizer = AutoTokenizer.from_pretrained(
    roberta_checkpoint
)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_checkpoint,
    num_labels=5,
    ignore_mismatched_sizes=True
)

deberta_model.to("cpu")
roberta_model.to("cpu")

deberta_model.eval()
roberta_model.eval()

print("Models loaded on CPU")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Models loaded on CPU


## Helper functions


In [6]:
def build_model_text(row, instruction_prefix=None):
    prompt = str(row["prompt"])

    if instruction_prefix:
        prompt = instruction_prefix + " " + prompt

    choices = " ".join(
        f"{letter}: {str(row[letter])}"
        for letter in option_letters
    )

    return f"{prompt} {choices}"



def predict_probabilities(
    row,
    tokenizer,
    model,
    instruction_prefix=None
):
    prompt = str(row["prompt"])

    if instruction_prefix is not None:
        prompt = instruction_prefix + " " + prompt

    choices = " ".join(
        f"{letter}: {str(row[letter])}"
        for letter in ["A", "B", "C", "D", "E"]
    )

    text = f"{prompt} {choices}"

    encoded = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    encoded = {
        key: value.to("cpu")
        for key, value in encoded.items()
    }

    model.to("cpu")
    model.eval()

    with torch.no_grad():
        outputs = model(**encoded)
        logits = outputs.logits

    probabilities = torch.softmax(logits, dim=-1)[0]

    return probabilities.cpu().numpy()

# Q1. DeBERTa inference for row index 25

Run the fine-tuned DeBERTa model independently and apply Softmax.

Report the option with the highest probability and its probability.


In [7]:
row_25 = train.iloc[25]

q1_deberta_probabilities = predict_probabilities(
    row_25,
    deberta_tokenizer,
    deberta_model
)

q1_top_index = int(np.argmax(q1_deberta_probabilities))
q1_top_option = id2label[q1_top_index]
q1_top_probability = float(
    q1_deberta_probabilities[q1_top_index]
)

print("Probabilities:", q1_deberta_probabilities)
print("Top option:", q1_top_option)
print("Top probability:", q1_top_probability)

Probabilities: [0.2168 0.2693 0.205  0.1782 0.1307]
Top option: B
Top probability: 0.269287109375


# Q2. Simple probability ensemble

Average the DeBERTa and RoBERTa probabilities for row index 25.


In [8]:
q2_roberta_probabilities = predict_probabilities(
    row_25,
    roberta_tokenizer,
    roberta_model
)

q2_average_probabilities = (
    q1_deberta_probabilities + q2_roberta_probabilities
) / 2.0

q2_top_option = id2label[int(np.argmax(q2_average_probabilities))]

print("DeBERTa probabilities:", q1_deberta_probabilities)
print("RoBERTa probabilities:", q2_roberta_probabilities)
print("Average probabilities:", q2_average_probabilities)
print("Top option:", q2_top_option)


DeBERTa probabilities: [0.2168 0.2693 0.205  0.1782 0.1307]
RoBERTa probabilities: [0.19298339 0.20105655 0.20133933 0.214451   0.19016963]
Average probabilities: [0.20489013 0.23517182 0.2031477  0.19633684 0.16045347]
Top option: B


# Q3. Weighted probability ensemble

Use:

- DeBERTa: 0.70
- RoBERTa: 0.30


In [9]:
def weighted_ensemble(
    deberta_probabilities,
    roberta_probabilities,
    deberta_weight=0.70,
    roberta_weight=0.30
):
    return (
        deberta_weight * deberta_probabilities
        + roberta_weight * roberta_probabilities
    )

In [10]:
q3_weighted_probabilities = weighted_ensemble(
    q1_deberta_probabilities,
    q2_roberta_probabilities,
    deberta_weight=0.70,
    roberta_weight=0.30
)

q3_top_option = id2label[int(np.argmax(q3_weighted_probabilities))]

print("Weighted probabilities:", q3_weighted_probabilities)
print("Top option:", q3_top_option)


Weighted probabilities: [0.20975049 0.2489156  0.20395648 0.1891522  0.14860362]
Top option: B


# Q4. Kaggle Top-3 prediction string

Rank all five options using the weighted probabilities and return the first three options separated by spaces.


In [11]:
import numpy as np

id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

def top3_string(probabilities):
    top_3_indices = np.argsort(probabilities)[::-1][:3]
    top_3_letters = [
        id2label[int(index)]
        for index in top_3_indices
    ]
    return " ".join(top_3_letters)

In [12]:
q4_prediction = top3_string(q3_weighted_probabilities)

print("Top-3 prediction:", q4_prediction)


Top-3 prediction: B A C


# Q5. Generate `submission.csv`

Run the weighted ensemble on every row of `test.csv`.

The output format is:

`id,prediction`


In [13]:
submission_predictions = []

for row_index in range(len(test)):
    row = test.iloc[row_index]

    deberta_probs = predict_probabilities(
        row,
        deberta_tokenizer,
        deberta_model
    )

    roberta_probs = predict_probabilities(
        row,
        roberta_tokenizer,
        roberta_model
    )

    final_probs = weighted_ensemble(
        deberta_probs,
        roberta_probs
    )

    submission_predictions.append(
        top3_string(final_probs)
    )

submission = pd.DataFrame({
    "id": test["id"],
    "prediction": submission_predictions
})

submission.to_csv("submission.csv", index=False)

print("Prediction rows:", len(submission))
submission.head()


Prediction rows: 500


,id,prediction
0,1,A B C
1,2,C B D
2,3,A C B
3,4,C B A
4,5,C D B


# Q6. Test-Time Augmentation

For the first 50 test rows, compare:

1. Original prompt inference
2. Average of original and instruction-augmented inference

Instruction prefix:

`Answer the following multiple-choice question carefully:`


In [14]:
instruction_prefix = (
    "Answer the following multiple-choice question carefully:"
)

q6_changed_count = 0
q6_results = []

for row_index in range(50):
    row = test.iloc[row_index]

    original_probs = predict_probabilities(
        row,
        deberta_tokenizer,
        deberta_model
    )

    augmented_probs = predict_probabilities(
        row,
        deberta_tokenizer,
        deberta_model,
        instruction_prefix=instruction_prefix
    )

    tta_probs = (original_probs + augmented_probs) / 2.0

    original_top1 = id2label[int(np.argmax(original_probs))]
    tta_top1 = id2label[int(np.argmax(tta_probs))]

    changed = original_top1 != tta_top1
    q6_changed_count += int(changed)

    q6_results.append({
        "row_index": row_index,
        "original_top1": original_top1,
        "tta_top1": tta_top1,
        "changed": changed
    })

q6_results = pd.DataFrame(q6_results)

print("Different Top-1 predictions:", q6_changed_count)
q6_results.head()


Different Top-1 predictions: 7


,row_index,original_top1,tta_top1,changed
0,0,A,A,False
1,1,C,C,False
2,2,A,A,False
3,3,C,C,False
4,4,C,C,False


# Q7. DeBERTa vs weighted ensemble Top-1

Compare the first 100 rows of `test.csv`.


In [15]:
q7_changed_count = 0
q7_results = []

for row_index in range(100):
    row = test.iloc[row_index]

    deberta_probs = predict_probabilities(
        row,
        deberta_tokenizer,
        deberta_model
    )

    roberta_probs = predict_probabilities(
        row,
        roberta_tokenizer,
        roberta_model
    )

    ensemble_probs = weighted_ensemble(
        deberta_probs,
        roberta_probs
    )

    deberta_top1 = id2label[int(np.argmax(deberta_probs))]
    ensemble_top1 = id2label[int(np.argmax(ensemble_probs))]

    changed = deberta_top1 != ensemble_top1
    q7_changed_count += int(changed)

    q7_results.append({
        "row_index": row_index,
        "deberta_top1": deberta_top1,
        "ensemble_top1": ensemble_top1,
        "changed": changed
    })

q7_results = pd.DataFrame(q7_results)

print("Different Top-1 predictions:", q7_changed_count)
q7_results.head()


Different Top-1 predictions: 3


,row_index,deberta_top1,ensemble_top1,changed
0,0,A,A,False
1,1,C,C,False
2,2,A,A,False
3,3,C,C,False
4,4,C,C,False


# Q8. Positive confidence gain

For the first 100 test rows:

`Confidence Gain = Ensemble Confidence - DeBERTa Confidence`


In [16]:
q8_positive_gain_count = 0
q8_results = []

for row_index in range(100):
    row = test.iloc[row_index]

    deberta_probs = predict_probabilities(
        row,
        deberta_tokenizer,
        deberta_model
    )

    roberta_probs = predict_probabilities(
        row,
        roberta_tokenizer,
        roberta_model
    )

    ensemble_probs = weighted_ensemble(
        deberta_probs,
        roberta_probs
    )

    deberta_confidence = float(np.max(deberta_probs))
    ensemble_confidence = float(np.max(ensemble_probs))
    confidence_gain = ensemble_confidence - deberta_confidence

    positive_gain = confidence_gain > 0
    q8_positive_gain_count += int(positive_gain)

    q8_results.append({
        "row_index": row_index,
        "deberta_confidence": deberta_confidence,
        "ensemble_confidence": ensemble_confidence,
        "confidence_gain": confidence_gain,
        "positive_gain": positive_gain
    })

q8_results = pd.DataFrame(q8_results)

print("Rows with positive confidence gain:", q8_positive_gain_count)
q8_results.head()


Rows with positive confidence gain: 0


,row_index,deberta_confidence,ensemble_confidence,confidence_gain,positive_gain
0,0,0.263916,0.242348,-0.021568,False
1,1,0.303223,0.271627,-0.031596,False
2,2,0.244019,0.229219,-0.014800,False
3,3,0.270752,0.249102,-0.021650,False
4,4,0.268311,0.248347,-0.019964,False


# Q9. Ordered Top-3 ranking changes

Compare the first 100 test rows using:

1. DeBERTa alone
2. Weighted ensemble


In [ ]:
q9_changed_count = 0
q9_results = []

for row_index in range(100):
    row = test.iloc[row_index]

    deberta_probs = predict_probabilities(
        row,
        deberta_tokenizer,
        deberta_model
    )

    roberta_probs = predict_probabilities(
        row,
        roberta_tokenizer,
        roberta_model
    )

    ensemble_probs = weighted_ensemble(
        deberta_probs,
        roberta_probs
    )

    deberta_top3 = top3_string(deberta_probs)
    ensemble_top3 = top3_string(ensemble_probs)

    changed = deberta_top3 != ensemble_top3
    q9_changed_count += int(changed)

    q9_results.append({
        "row_index": row_index,
        "deberta_top3": deberta_top3,
        "ensemble_top3": ensemble_top3,
        "changed": changed
    })

q9_results = pd.DataFrame(q9_results)

print("Rows with ordered Top-3 changes:", q9_changed_count)
q9_results.head()


# Q10. MAP@3 on the first 100 validation samples

The competition `test.csv` has no answer column, so this notebook uses the first 100 rows of `train.csv` as the validation samples.

For each row:

- Generate the weighted ensemble Top-3 prediction
- Compare it with the true answer
- Calculate AP@3
- Average the 100 AP@3 values


In [ ]:
q10_scores = []
q10_results = []

for row_index in range(100):
    row = train.iloc[row_index]

    deberta_probs = predict_probabilities(
        row,
        deberta_tokenizer,
        deberta_model
    )

    roberta_probs = predict_probabilities(
        row,
        roberta_tokenizer,
        roberta_model
    )

    ensemble_probs = weighted_ensemble(
        deberta_probs,
        roberta_probs
    )

    prediction = top3_string(ensemble_probs)
    correct_letter = str(row["answer"]).strip()

    row_score = average_precision_at_3(
        prediction,
        correct_letter
    )

    q10_scores.append(row_score)

    q10_results.append({
        "row_index": row_index,
        "correct_answer": correct_letter,
        "prediction": prediction,
        "ap_at_3": row_score
    })

q10_map3 = float(np.mean(q10_scores))
q10_results = pd.DataFrame(q10_results)

print("MAP@3:", round(q10_map3, 4))
q10_results.head()
